# Pipeline de reconocimiento de marcha en Google Colab

Ejecuta el flujo del repositorio en la nube: importar el dataset compartido en Drive, generar manifiesto, recortar caminatas, extraer siluetas/keypoints, evaluar y exportar los assets móviles.

**Este proyecto usa GaitBase preentrenado; no contiene entrenamiento ni fine-tuning de GaitBase.** El notebook ejecuta inferencia y construye/evalúa la galería. No es necesario volver a entrenar el modelo para este flujo.

En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. Ejecuta las celdas en orden. La carpeta de Drive se solicita durante la ejecución y su enlace no queda guardado en el notebook.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import auth, drive

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de inferencia: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

auth.authenticate_user()
drive.mount("/content/drive")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/GaitRecognitionColab")
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Importar el dataset desde Drive

Pega el enlace de la carpeta compartida. La cuenta autenticada debe tener permiso de lectura. El contenido se copia al disco temporal de Colab para procesarlo más rápido; los resultados se guardan en `MyDrive/GaitRecognitionColab`.

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

DATASET_FOLDER_URL = input("Pega el enlace de la carpeta del dataset en Google Drive: ").strip()
folder_match = re.search(r"/folders/([A-Za-z0-9_-]+)", DATASET_FOLDER_URL)
DATASET_FOLDER_ID = folder_match.group(1) if folder_match else DATASET_FOLDER_URL
if not re.fullmatch(r"[A-Za-z0-9_-]+", DATASET_FOLDER_ID):
    raise ValueError("El enlace no contiene un ID válido de carpeta de Drive.")

drive_api = build("drive", "v3", cache_discovery=False)
root_info = drive_api.files().get(
    fileId=DATASET_FOLDER_ID,
    fields="id,name,mimeType",
    supportsAllDrives=True,
).execute()
if root_info["mimeType"] != "application/vnd.google-apps.folder":
    raise ValueError("El enlace debe apuntar a una carpeta de Google Drive.")

DATASET_ROOT = Path("/content/gait_dataset") / root_info["name"]
DATASET_ROOT.mkdir(parents=True, exist_ok=True)
visited_folders = set()

def download_drive_folder(folder_id, local_dir):
    if folder_id in visited_folders:
        return
    visited_folders.add(folder_id)
    page_token = None
    while True:
        response = drive_api.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            pageSize=1000,
            pageToken=page_token,
            fields="nextPageToken,files(id,name,mimeType)",
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
        ).execute()
        for item in response.get("files", []):
            target = local_dir / item["name"]
            if item["mimeType"] == "application/vnd.google-apps.folder":
                target.mkdir(parents=True, exist_ok=True)
                download_drive_folder(item["id"], target)
            elif item["mimeType"].startswith("application/vnd.google-apps."):
                print(f"Omitido documento nativo de Google: {item['name']}")
            else:
                request = drive_api.files().get_media(
                    fileId=item["id"], supportsAllDrives=True
                )
                with target.open("wb") as output_file:
                    downloader = MediaIoBaseDownload(output_file, request, chunksize=16 * 1024 * 1024)
                    done = False
                    while not done:
                        _, done = downloader.next_chunk()
                print(f"Descargado: {target.relative_to(DATASET_ROOT)}")
        page_token = response.get("nextPageToken")
        if not page_token:
            break

download_drive_folder(DATASET_FOLDER_ID, DATASET_ROOT)
video_count = sum(1 for _ in DATASET_ROOT.rglob("*.mp4"))
if video_count == 0:
    raise RuntimeError(
        f"No se encontraron MP4 dentro de {DATASET_ROOT}. Comprueba que la carpeta compartida "
        "contenga directamente las carpetas de sujetos y tomas."
    )
print(f"Dataset listo: {video_count} videos en {DATASET_ROOT}")

## 2. Preparar el código y dependencias

Se instala el repositorio y OpenGait dentro de la sesión de Colab. Si estos cambios todavía no están publicados en GitHub, primero súbelos; de lo contrario Colab clonará la versión remota anterior.

In [ ]:
REPO_URL = "https://github.com/IDKGabriel15/gait-recognition-proyecto.git"
REPO_DIR = Path("/content/gait-recognition-proyecto")
OPENGAIT_DIR = REPO_DIR / "third_party" / "OpenGait"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if not OPENGAIT_DIR.exists():
    OPENGAIT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ShiqiYu/OpenGait.git", str(OPENGAIT_DIR)],
        check=True,
    )

packages = [
    "ultralytics", "opencv-python-headless", "onnx", "onnxruntime", "scipy",
    "scikit-learn", "pyyaml", "tqdm", "einops", "kornia", "tensorboard",
    "google-api-python-client",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(OPENGAIT_DIR))
print("Repositorio y dependencias listos.")

## 3. Configurar y crear el manifiesto

Por defecto se procesa todo el dataset. Para una prueba corta, fija `MAX_SUBJECTS` o `MAX_TAKES_PER_SUBJECT` a un entero pequeño y vuelve a ejecutar las celdas de procesamiento.

In [ ]:
WORK_ROOT = Path("/content/gait_work")
CONFIG_DIR = WORK_ROOT / "configs"
TRIMMED_ROOT = WORK_ROOT / "dataset_trimmed"
FEATURES_ROOT = WORK_ROOT / "outputs" / "features"
VISUALS_ROOT = WORK_ROOT / "outputs" / "visuals"
WEIGHTS_DIR = WORK_ROOT / "weights"
MOBILE_ASSETS = WORK_ROOT / "mobile_assets"
MANIFEST_PATH = CONFIG_DIR / "dataset_manifest.json"
GALLERY_PKL = WORK_ROOT / "outputs" / "gallery_db.pkl"
CHECKPOINT_PATH = WEIGHTS_DIR / "GaitBase_DA-180000.pt"
OVERWRITE = False
MAX_SUBJECTS = None
MAX_TAKES_PER_SUBJECT = 0

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
for path in (TRIMMED_ROOT, FEATURES_ROOT, VISUALS_ROOT, WEIGHTS_DIR, MOBILE_ASSETS):
    path.mkdir(parents=True, exist_ok=True)

import phase0_manifest
phase0_manifest.DATASET_ROOT = DATASET_ROOT
phase0_manifest.OUTPUT_MANIFEST = MANIFEST_PATH
manifest = phase0_manifest.build_manifest()
print(manifest["summary"])
if manifest["summary"]["num_subjects"] == 0:
    raise RuntimeError("El manifiesto no encontró sujetos; verifica la estructura del dataset.")

## 4. Detectar y recortar las pasadas de caminata

YOLOv8 detecta el sujeto y recorta los segmentos activos. Esta etapa puede tardar bastante; deja `OVERWRITE=False` para reutilizar recortes existentes al reanudar la sesión.

In [ ]:
from tqdm.auto import tqdm
from ultralytics import YOLO
import phase1_trim

phase1_trim.MANIFEST_PATH = MANIFEST_PATH
phase1_trim.TRIMMED_ROOT = TRIMMED_ROOT
detector = YOLO("yolov8n.pt")
trimmed_count = 0
groups = [("subjects", manifest["subjects"]), ("external_test", manifest["external_test"])]
for group_name, group_dict in groups:
    subjects = list(sorted(group_dict.items()))
    if MAX_SUBJECTS is not None:
        subjects = subjects[:MAX_SUBJECTS]
    for subject, subject_data in tqdm(subjects, desc=f"Recortes {group_name}"):
        takes = subject_data if group_name == "external_test" else subject_data["gallery"] + subject_data["probe"]
        if MAX_TAKES_PER_SUBJECT:
            takes = takes[:MAX_TAKES_PER_SUBJECT]
        for take in takes:
            out_dir = TRIMMED_ROOT / group_name / subject / take["take_name"]
            result = phase1_trim.process_take(take, out_dir, detector, overwrite=OVERWRITE)
            trimmed_count += int(result is not None)
print(f"Tomas recortadas o reutilizadas: {trimmed_count}")

## 5. Extraer siluetas y keypoints

Se generan `silhouettes.npy` y `keypoints.npy` para cada pasada y ángulo. Los pesos YOLO se descargan automáticamente si no están en la sesión.

In [ ]:
import phase2_extract_and_visualize
phase2_extract_and_visualize.TRIMMED_ROOT = TRIMMED_ROOT
phase2_extract_and_visualize.FEATURES_ROOT = FEATURES_ROOT
phase2_extract_and_visualize.VISUALS_ROOT = VISUALS_ROOT
seg_model = YOLO("yolov8n-seg.pt")
pose_model = YOLO("yolov8n-pose.pt")
clips = []
for group_name in ("subjects", "external_test"):
    group_root = TRIMMED_ROOT / group_name
    if not group_root.exists():
        continue
    subject_dirs = sorted(p for p in group_root.iterdir() if p.is_dir())
    if MAX_SUBJECTS is not None:
        subject_dirs = subject_dirs[:MAX_SUBJECTS]
    for subject_dir in subject_dirs:
        for take_dir in sorted(p for p in subject_dir.iterdir() if p.is_dir()):
            pass_dirs = sorted(p for p in take_dir.iterdir() if p.is_dir() and p.name.startswith("pass_"))
            for pass_dir in pass_dirs:
                for video in sorted(pass_dir.glob("*.mp4")):
                    clips.append((group_name, subject_dir.name, take_dir.name, pass_dir.name, video))

for group_name, subject, take_name, pass_name, video in tqdm(clips, desc="Extracción de características"):
    feat_dir = FEATURES_ROOT / group_name / subject / take_name / pass_name / video.stem
    phase2_extract_and_visualize.process_video(
        video, feat_dir, seg_model, pose_model, overwrite=OVERWRITE
    )
print(f"Clips procesados: {len(clips)}")

## 6. Construir y evaluar la galería GaitBase

Se descargan los pesos preentrenados si hacen falta. El extractor aprovecha la GPU cuando Colab la ofrece; esta etapa **no actualiza los pesos**. La división gallery/probe sigue la regla del proyecto: tomas 1–7 para galería y desde la 8 para probe.

In [ ]:
import json
import pickle
import download_weights
download_weights.OUT_PATH = CHECKPOINT_PATH
download_weights.main()

from phase3_gait_engine import GaitBaseInference
import phase3_evaluate
phase3_evaluate.MANIFEST_PATH = MANIFEST_PATH
phase3_evaluate.FEATURES_ROOT = FEATURES_ROOT
phase3_evaluate.CKPT_PATH = CHECKPOINT_PATH
phase3_evaluate.GALLERY_DB_PATH = GALLERY_PKL
gait_model = GaitBaseInference(CHECKPOINT_PATH).to(DEVICE)
gallery_db = phase3_evaluate.build_gallery(manifest, gait_model)
GALLERY_PKL.parent.mkdir(parents=True, exist_ok=True)
with GALLERY_PKL.open("wb") as f:
    pickle.dump(gallery_db, f)
print(f"Galería: {len(gallery_db)} sujetos, {sum(map(len, gallery_db.values()))} entradas")
phase3_evaluate.evaluate_probes(manifest, gallery_db, gait_model)

## 7. Exportar ONNX y galería para Android

Genera el ONNX con salida por franja, YOLO ONNX y los binarios de galería con media/normalización compatibles con el `GaitEngine.kt` actualizado.

In [ ]:
import export_mobile
export_mobile.CKPT_PATH = CHECKPOINT_PATH
export_mobile.GALLERY_PKL = GALLERY_PKL
export_mobile.OUT_DIR = MOBILE_ASSETS
sample_candidates = sorted(FEATURES_ROOT.rglob("silhouettes.npy"))
if not sample_candidates:
    raise RuntimeError("No hay silhouettes.npy para verificar la exportación ONNX.")
export_mobile.SAMPLE_SIL_PATH = sample_candidates[0]
export_mobile.export_gaitbase_onnx(MOBILE_ASSETS / "gaitbase_mobile.onnx")
export_mobile.export_yolo_onnx()
export_mobile.export_gallery_binary()
print("Assets listos:", sorted(p.name for p in MOBILE_ASSETS.iterdir()))

## 8. Guardar resultados en Google Drive

Se guardan el manifiesto, la galería evaluada y los assets móviles en `MyDrive/GaitRecognitionColab`. Los videos recortados y las características intermedias permanecen en el disco temporal de Colab para no duplicar automáticamente un dataset posiblemente grande.

In [ ]:
import zipfile

results_dir = DRIVE_OUTPUT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(MANIFEST_PATH, results_dir / MANIFEST_PATH.name)
shutil.copy2(GALLERY_PKL, results_dir / GALLERY_PKL.name)
drive_mobile_dir = results_dir / "mobile_assets"
shutil.copytree(MOBILE_ASSETS, drive_mobile_dir, dirs_exist_ok=True)
zip_path = results_dir / "mobile_assets.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for asset in sorted(drive_mobile_dir.iterdir()):
        if asset.is_file():
            archive.write(asset, arcname=asset.name)
print(f"Resultados guardados en: {results_dir}")
print(f"Paquete móvil: {zip_path}")

### Reanudar una sesión

Los archivos temporales de Colab se borran al terminar la sesión. Para evitar repetir etapas, copia `dataset_trimmed/` y `outputs/features/` del directorio `/content/gait_work` a Drive antes de cerrar, y restáuralos en el mismo sitio al abrir otra sesión. La galería y los assets finales ya quedan persistidos automáticamente.